# Train the SAT-selector GNN on a free Colab GPU

**One click:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

This notebook clones the project, installs it, and runs `hsat pipeline`: it downloads the ASlib
scenarios and CNF instances, builds the literal-clause graphs, trains the encoder inside
cross-validation (supervised per fold, contrastive once, and a family-held-out run), and draws
the figures. The GPU profile trains on SAT18-EXP and SAT03-16_INDU.

**Sessions time out.** Everything is resumable: with `USE_DRIVE = True` the data, graphs and
every trained fold are kept in your Google Drive, and *Run all* again continues where it stopped.

In [ ]:
# ---- settings ---------------------------------------------------------------------------
REPO = "zishaan1911/hybrid-sat-selector"
BRANCH = "main"
USE_DRIVE = True        # keep data + trained folds in Google Drive so a timeout loses nothing
SCENARIOS = "sat18,indu"  # "sat18" alone takes much less time
# Private repository? Add a Colab secret named GITHUB_TOKEN (key icon in the left bar).

In [ ]:
import os, subprocess
from pathlib import Path

WORK = Path("/content/hybrid-sat-selector")   # the code: fast local disk
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
if not (WORK / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(WORK)], check=True)
os.chdir(WORK)

if USE_DRIVE:
    # data/ holds the CNFs, graphs and every trained fold. Keeping it in Drive means a
    # timed-out session resumes: finished folds are reloaded, not retrained.
    from google.colab import drive
    drive.mount("/content/drive")
    stored = Path("/content/drive/MyDrive/hybrid-sat-selector-data")
    stored.mkdir(parents=True, exist_ok=True)
    if not Path("data").is_symlink():
        subprocess.run(["rm", "-rf", "data"], check=True)
        Path("data").symlink_to(stored)
print("code in", WORK, "| data in", Path("data").resolve())
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > GPU"


In [ ]:
# Installs into Colab's own Python (its PyTorch already has CUDA), then runs every step.
!python scripts/setup.py --no-venv --scenarios {SCENARIOS}

## Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 160)
for csv in sorted(Path("experiments/e10_training/results").glob("*.csv")):
    frame = pd.read_csv(csv)
    print(f"\n=== {csv.stem}")
    display(frame[["selector", "par10", "gap_closed", "par10_charged", "gap_closed_charged",
                   "accuracy"]].round(3))
for png in sorted(Path("docs/figures/results").glob("*.png")):
    if any(key in png.name for key in ("supervised", "contrastive")):
        display(Image(str(png), width=720))

## Share the results

Commit the result tables, the run registry and the figures (a few hundred KB; data and
checkpoints stay out of git), or download them as a zip and commit from your own machine.

In [ ]:
!zip -qr /content/e10_results.zip experiments/e10_training experiments/runs.csv docs/figures/results
from google.colab import files
files.download("/content/e10_results.zip")